In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 9
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
14.422091841846251

Trial 1 =========================================
14.404885915872025

Trial 2 =========================================
17.5256114251161

Trial 3 =========================================
17.530955195856777

Trial 4 =========================================
15.882587091866217



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/botorch/optim/optimize.py:331: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  generated_initial_conditions = opt_inputs.get_ic_generator()(


Trial 5 =========================================
14.040531111783226

Trial 6 =========================================
14.438794109317483

Trial 7 =========================================
18.82165375276476

Trial 8 =========================================
13.431819515575803

Trial 9 =========================================
15.314720851327163

Trial 10 =========================================
15.760288147229723

Trial 11 =========================================
15.13199105308079

Trial 12 =========================================
13.354292498682558



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 13 =========================================
15.995027231406894

Trial 14 =========================================
13.880628778450264

Trial 15 =========================================
14.45482887923959



...

Trial 68 =========================================
15.747954839030822

Trial 69 =========================================
13.771182253094246

Trial 70 =========================================
14.28783880694781

Trial 71 =========================================
14.929672496505983

Trial 72 =========================================
15.939403232363688

Trial 73 =========================================
15.297639445894585

Trial 74 =========================================
14.242639192012422

Trial 75 =========================================
14.270652286241969

Trial 76 =========================================
15.507858656650145

Trial 77 =========================================
15.83383532597144

Trial 78 =========================================
14.444617615218366



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 79 =========================================
15.907320776277295

Trial 80 =========================================
15.190269185816906

Trial 81 =========================================
15.397055740139395

Trial 82 =========================================
14.399311202943839

Trial 83 =========================================
14.279009896348917

Trial 84 =========================================
15.351954929351201

Trial 85 =========================================
14.83811818703987

Trial 86 =========================================
18.82076188901499

Trial 87 =========================================
14.44677780411858

Trial 88 =========================================
16.2432538696434

Trial 89 =========================================
15.929222010399386

Trial 90 =========================================
14.624923005676065

Trial 91 =========================================
14.28125645554354

Trial 92 =========================================
15.394010680216962

Trial 93 ===

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/botorch/optim/optimize.py:331: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  generated_initial_conditions = opt_inputs.get_ic_generator()(


Trial 96 =========================================
14.333722845485855

Trial 97 =========================================
16.91044167849755

Trial 98 =========================================
15.973533411515017

Trial 99 =========================================
15.37637164397215

[14.42209184 14.40488592 17.52561143 17.5309552  15.88258709 14.04053111
 14.43879411 18.82165375 13.43181952 15.31472085 15.76028815 15.13199105
 13.3542925  15.99502723 13.88062878 14.45482888 15.9707302  16.2434748
 14.33836625 15.3057473  15.34707719 14.26597454 17.51380226 16.21966147
 14.80704821 18.80404345 15.3431474  13.95658055 15.39094445 17.52385614
 13.88716398 15.36361124 17.47236633 15.37528169 15.92922201 15.38968371
 14.16148858 16.10727869 16.23932069 17.42983089 14.06442793 13.18570898
 16.16567807 16.24294154 14.45659063 15.16667554 14.6898356  17.56602066
 14.27927482 15.98611195 18.81469559 15.38437979 16.16020447 13.61024471
 15.36552771 15.44244004 15.38959196 17.54423985 14.59955835 1

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.82165375276476
Avg = 15.458103450826522
Std = 1.2661396160347453


In [6]:
print(y_max_arr.tolist())

[14.422091841846251, 14.404885915872025, 17.5256114251161, 17.530955195856777, 15.882587091866217, 14.040531111783226, 14.438794109317483, 18.82165375276476, 13.431819515575803, 15.314720851327163, 15.760288147229723, 15.13199105308079, 13.354292498682558, 15.995027231406894, 13.880628778450264, 14.45482887923959, 15.970730198545885, 16.243474796202793, 14.338366254574186, 15.305747295741522, 15.347077191129786, 14.265974544316736, 17.513802259144594, 16.219661473005175, 14.807048213952385, 18.80404345191846, 15.343147397064813, 13.956580550439073, 15.39094444644892, 17.5238561449111, 13.887163980342745, 15.363611241271537, 17.47236632867842, 15.37528169448081, 15.929222010399386, 15.389683714565365, 14.161488584148861, 16.107278689539935, 16.239320692773525, 17.429830893414998, 14.064427928733323, 13.185708975952325, 16.165678065297076, 16.24294153880792, 14.456590630576208, 15.166675535344059, 14.689835601111664, 17.566020656859934, 14.279274822820158, 15.986111950280673, 18.81469558

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-9.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)